<a href="https://colab.research.google.com/github/ron360/DS_demo_website/blob/main/112306073_%E8%B3%87%E7%AE%A1%E4%B8%89_%E6%9D%8E%E5%B2%B3%E8%9E%8D_%E5%81%A5%E4%BA%BA%E5%BC%8F%E6%80%9D%E8%80%83CoT_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**重點統整**

我的主題是健身人的思考回覆，第一步我先讓AI 思考事件如何從訓練、恢復、飲食、紀律、長期進步等健身相關面向去連結，第二步再從中選出合理的敘述，並用健身專家的語氣表達

#### 1. 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [ ]:
import os
from google.colab import userdata

In [ ]:

api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
provider = "groq"
model = "openai/gpt-oss-120b"

In [ ]:
!pip install aisuite[all]

### 2. 使用 AISuite 的準備

In [ ]:
import aisuite as ai

In [ ]:
provider_planner = "groq"
model_planner="openai/gpt-oss-120b"

provider_writer = "groq"
model_writer = "openai/gpt-oss-120b"


In [ ]:
def reply(system="請用台灣習慣的中文回覆。",
          prompt="Hi",
          provider="groq",
          model="openai/gpt-oss-120b"
          ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content

####  3. 打造二階段

In [ ]:
system_planner = "請用台灣習慣的中文回應: 你是一位健身教練，習慣用「訓練、恢復、飲食、紀律、長期進步」這幾個面向來看事情。使用者會說一件跟生活或訓練有關的事情（可能看起來有點衰、有點卡關、或只是日常小事）。請你站在健身人士的角度，以列點的方式思考出「為什麼這其實對我的訓練人生是好事」的理由。"
system_writer = "請用台灣習慣的中文回應: 你是一位熱愛訓練的健身 KOL，風格有點嘴砲但正向，會把生活事件全部解讀成「變強養分」。請挑選三個特別有說服力的理由加入敘述，並在結尾加上一句「反正一切都是為了變強 💪」 作為收尾。"

In [ ]:
def post(prompt):
    # Step 1: 根據不同健身相關面向思考理由
    planning_prompt = f"使用者說：{prompt}。請幫我想如何解釋是變強的理由。"
    reasons = reply(system_planner, planning_prompt,
                          provider = provider_planner,
                          model = model_planner
                          )

    # Step 2: 選出三項理由，寫成完整敘述
    generation_prompt = f"這是我想到的五個理由：\n{reasons}\n\n請從中選出三個有說服力理由，然後根據它寫一段完整健身人式發言，不要列點。"
    final_post = reply(system_writer, generation_prompt,
                       provider = provider_writer,
                       model = model_writer
                       )

    return reasons, final_post

### 4. 用 Gradio 打造你的對話機器人 Web App!

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("### 💪 健人思考產生器 💪")
    gr.Markdown("請輸入一件你最近發生的日常事件，讓 AI 幫你用健人的方式重新詮釋！")
    user_input = gr.Textbox(label="今天發生的事情是…")
    btn = gr.Button("生成健人回覆 ✨")

    with gr.Row():
        out1 = gr.Textbox(label="🧠 變強理由）")
        out2 = gr.Textbox(label="📣 最終健人回覆")

    btn.click(post, inputs=[user_input], outputs=[out1, out2])

In [ ]:
demo.launch(share=True, debug=True)